In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "litsearch"


In [2]:
from src.rerank.coarse import run_coarse_reranking

ValueError: DEEPSEEK_API_KEY is not set

In [3]:
import json

def dummy_coarse_llm(prompt: str) -> str:
    # очень простой "ранкер": возвращаем doc_id в исходном порядке
    ids = []
    for line in prompt.splitlines():
        if line.strip().startswith("Document"):
            continue
    # для простоты вернём пустой список или фиктивный порядок
    return json.dumps([])


In [4]:
run_coarse_reranking(
    ie_path=DATA_DIR / "ie_documents.jsonl",
    queries_path=DATA_DIR / "queries.jsonl",
    output_path=DATA_DIR / "coarse_rerank.jsonl",
    llm_call_fn=dummy_coarse_llm,
    top_k=20,
)


In [5]:
!head -n 3 ../data/processed/litsearch/coarse_rerank.jsonl | jq .


{
  "query_id": 0,
  "ranked_doc_ids": []
}
{
  "query_id": 1,
  "ranked_doc_ids": []
}
{
  "query_id": 2,
  "ranked_doc_ids": []
}


In [6]:
!head -n 3 ../data/processed/litsearch/queries.jsonl | jq .


{
  "qid": 0,
  "text": "Are there any research papers on methods to compress large-scale language models using task-agnostic knowledge distillation techniques?",
  "relevant_doc_ids": [
    202719327
  ]
}
{
  "qid": 1,
  "text": "Are there any resources available for translating Tunisian Arabic dialect that contain both manually translated comments by native speakers and additional data augmented through methods like segmentation at stop words level?",
  "relevant_doc_ids": [
    227231792
  ]
}
{
  "qid": 2,
  "text": "Are there any studies that explore post-hoc techniques for hallucination detection at both the token- and sentence-level in neural sequence generation tasks?",
  "relevant_doc_ids": [
    226254579,
    204976362
  ]
}


In [7]:
from src.rerank.checks import (
    check_non_empty,
    check_all_queries_present,
    check_non_empty_rankings,
    check_doc_ids_exist,
    check_ranking_lengths,
)

In [9]:
results = []

results.append(
    check_non_empty(DATA_DIR / "coarse_rerank.jsonl")
)

results.append(
    check_all_queries_present(
        DATA_DIR / "coarse_rerank.jsonl",
        DATA_DIR / "queries.jsonl",
    )
)

results.append(
    check_non_empty_rankings(DATA_DIR / "coarse_rerank.jsonl")
)

results.append(
    check_doc_ids_exist(
        DATA_DIR / "coarse_rerank.jsonl",
        DATA_DIR / "corpus.jsonl",
    )
)

results.append(
    check_ranking_lengths(
        DATA_DIR / "coarse_rerank.jsonl",
        max_k=20,
    )
)

results

[{'check': 'non_empty_file', 'num_lines': 597, 'ok': True},
 {'check': 'all_queries_present',
  'num_queries': 597,
  'num_found': 597,
  'missing_query_ids': [],
  'ok': True},
 {'check': 'non_empty_rankings',
  'num_empty': 597,
  'empty_query_ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
  'ok': False},
 {'check': 'doc_ids_exist_in_corpus',
  'num_queries_with_missing': 0,
  'examples': {},
  'ok': True},
 {'check': 'ranking_length',
  'num_too_long': 0,
  'too_long_query_ids': [],
  'ok': True}]

In [10]:
for r in results:
    print(f"[{r['check']}] OK = {r['ok']}")
    for k, v in r.items():
        if k not in {"check", "ok"}:
            print(f"  {k}: {v}")
    print()


[non_empty_file] OK = True
  num_lines: 597

[all_queries_present] OK = True
  num_queries: 597
  num_found: 597
  missing_query_ids: []

[non_empty_rankings] OK = False
  num_empty: 597
  empty_query_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

[doc_ids_exist_in_corpus] OK = True
  num_queries_with_missing: 0
  examples: {}

[ranking_length] OK = True
  num_too_long: 0
  too_long_query_ids: []

